In [1]:
import sys
import socket
import platform
from pathlib import Path

print("Computer:", socket.gethostname())
print("OS:", platform.platform())
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Computer: MSI
OS: Windows-11-10.0.26300-SP0
Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Python executable: C:\Users\tscou\AppData\Local\Python\pythoncore-3.14-64\python.exe
Working directory: C:\Users\tscou\capstone


In [10]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

data_path = (
    project_root
    / "data"
    / "raw"
    / "all_tickets_processed_improved_v3.csv"
)

print("Current directory:", Path.cwd())
print("Project root:", project_root)
print("Dataset path:", data_path)
print("Dataset exists:", data_path.exists())

df = pd.read_csv(data_path)

print("Shape:", df.shape)
display(df.head())

Current directory: C:\Users\tscou\capstone
Project root: C:\Users\tscou\capstone
Dataset path: C:\Users\tscou\capstone\data\raw\all_tickets_processed_improved_v3.csv
Dataset exists: True
Shape: (47837, 2)


,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous


In [11]:
df = df[["Document", "Topic_group"]].copy()

df = df.dropna(subset=["Document", "Topic_group"])

df["Document"] = df["Document"].astype(str).str.strip()
df["Topic_group"] = df["Topic_group"].astype(str).str.strip()

df = df[
    (df["Document"] != "") &
    (df["Topic_group"] != "")
]

print("Rows after cleaning:", len(df))
print("Number of classes:", df["Topic_group"].nunique())

print("\nClass distribution:")
display(df["Topic_group"].value_counts())

Rows after cleaning: 47837
Number of classes: 8

Class distribution:


Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

X = df["Document"]
y = df["Topic_group"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
display(y_train.value_counts())

print("\nTesting class distribution:")
display(y_test.value_counts())

Training samples: 38269
Testing samples: 9568

Training class distribution:


Topic_group
Hardware                 10893
HR Support                8732
Access                    5700
Miscellaneous             5648
Storage                   2222
Purchase                  1971
Internal Project          1695
Administrative rights     1408
Name: count, dtype: int64


Testing class distribution:


Topic_group
Hardware                 2724
HR Support               2183
Access                   1425
Miscellaneous            1412
Storage                   555
Purchase                  493
Internal Project          424
Administrative rights     352
Name: count, dtype: int64

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import time

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000
)

start_time = time.perf_counter()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

tfidf_time = time.perf_counter() - start_time

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))
print(f"TF-IDF processing time: {tfidf_time:.2f} seconds")

Training matrix shape: (38269, 50000)
Testing matrix shape: (9568, 50000)
Vocabulary size: 50000
TF-IDF processing time: 3.23 seconds


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
import time

log_reg = LogisticRegression(
    max_iter=2000,
    random_state=42
)

train_start = time.perf_counter()

log_reg.fit(X_train_tfidf, y_train)

training_time = time.perf_counter() - train_start

predict_start = time.perf_counter()

log_reg_predictions = log_reg.predict(X_test_tfidf)

inference_time = time.perf_counter() - predict_start

accuracy = accuracy_score(y_test, log_reg_predictions)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test,
    log_reg_predictions,
    average="macro",
    zero_division=0
)

precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test,
    log_reg_predictions,
    average="weighted",
    zero_division=0
)

print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1:        {f1_macro:.4f}")
print(f"Weighted F1:     {f1_weighted:.4f}")

print()
print(f"Training time:   {training_time:.2f} seconds")
print(f"Inference time:  {inference_time:.4f} seconds")

print("\nClassification Report:\n")
print(
    classification_report(
        y_test,
        log_reg_predictions,
        zero_division=0
    )
)

LOGISTIC REGRESSION RESULTS
Accuracy:        0.8508
Macro Precision: 0.8908
Macro Recall:    0.8146
Macro F1:        0.8470
Weighted F1:     0.8508

Training time:   11.22 seconds
Inference time:  0.0047 seconds

Classification Report:

                       precision    recall  f1-score   support

               Access       0.91      0.87      0.89      1425
Administrative rights       0.90      0.61      0.73       352
           HR Support       0.86      0.86      0.86      2183
             Hardware       0.78      0.89      0.84      2724
     Internal Project       0.92      0.79      0.85       424
        Miscellaneous       0.83      0.82      0.82      1412
             Purchase       0.98      0.86      0.92       493
              Storage       0.95      0.81      0.87       555

             accuracy                           0.85      9568
            macro avg       0.89      0.81      0.85      9568
         weighted avg       0.86      0.85      0.85      9568



In [16]:
from sklearn.svm import LinearSVC

linear_svm = LinearSVC(
    random_state=42
)

train_start = time.perf_counter()

linear_svm.fit(X_train_tfidf, y_train)

svm_training_time = time.perf_counter() - train_start

predict_start = time.perf_counter()

svm_predictions = linear_svm.predict(X_test_tfidf)

svm_inference_time = time.perf_counter() - predict_start

svm_accuracy = accuracy_score(y_test, svm_predictions)

svm_precision_macro, svm_recall_macro, svm_f1_macro, _ = (
    precision_recall_fscore_support(
        y_test,
        svm_predictions,
        average="macro",
        zero_division=0
    )
)

_, _, svm_f1_weighted, _ = precision_recall_fscore_support(
    y_test,
    svm_predictions,
    average="weighted",
    zero_division=0
)

print("LINEAR SVM RESULTS")
print("=" * 50)

print(f"Accuracy:        {svm_accuracy:.4f}")
print(f"Macro Precision: {svm_precision_macro:.4f}")
print(f"Macro Recall:    {svm_recall_macro:.4f}")
print(f"Macro F1:        {svm_f1_macro:.4f}")
print(f"Weighted F1:     {svm_f1_weighted:.4f}")

print()
print(f"Training time:   {svm_training_time:.2f} seconds")
print(f"Inference time:  {svm_inference_time:.4f} seconds")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        svm_predictions,
        zero_division=0
    )
)

LINEAR SVM RESULTS
Accuracy:        0.8587
Macro Precision: 0.8797
Macro Recall:    0.8456
Macro F1:        0.8614
Weighted F1:     0.8587

Training time:   1.36 seconds
Inference time:  0.0058 seconds

Classification Report:

                       precision    recall  f1-score   support

               Access       0.90      0.90      0.90      1425
Administrative rights       0.86      0.71      0.78       352
           HR Support       0.86      0.86      0.86      2183
             Hardware       0.82      0.86      0.84      2724
     Internal Project       0.90      0.84      0.87       424
        Miscellaneous       0.82      0.83      0.82      1412
             Purchase       0.95      0.88      0.92       493
              Storage       0.92      0.87      0.89       555

             accuracy                           0.86      9568
            macro avg       0.88      0.85      0.86      9568
         weighted avg       0.86      0.86      0.86      9568



In [17]:
from sklearn.naive_bayes import MultinomialNB

naive_bayes = MultinomialNB()

train_start = time.perf_counter()

naive_bayes.fit(X_train_tfidf, y_train)

nb_training_time = time.perf_counter() - train_start

predict_start = time.perf_counter()

nb_predictions = naive_bayes.predict(X_test_tfidf)

nb_inference_time = time.perf_counter() - predict_start

nb_accuracy = accuracy_score(y_test, nb_predictions)

nb_precision_macro, nb_recall_macro, nb_f1_macro, _ = (
    precision_recall_fscore_support(
        y_test,
        nb_predictions,
        average="macro",
        zero_division=0
    )
)

_, _, nb_f1_weighted, _ = precision_recall_fscore_support(
    y_test,
    nb_predictions,
    average="weighted",
    zero_division=0
)

print("MULTINOMIAL NAIVE BAYES RESULTS")
print("=" * 50)

print(f"Accuracy:        {nb_accuracy:.4f}")
print(f"Macro Precision: {nb_precision_macro:.4f}")
print(f"Macro Recall:    {nb_recall_macro:.4f}")
print(f"Macro F1:        {nb_f1_macro:.4f}")
print(f"Weighted F1:     {nb_f1_weighted:.4f}")

print()
print(f"Training time:   {nb_training_time:.2f} seconds")
print(f"Inference time:  {nb_inference_time:.4f} seconds")

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        nb_predictions,
        zero_division=0
    )
)

MULTINOMIAL NAIVE BAYES RESULTS
Accuracy:        0.7502
Macro Precision: 0.8859
Macro Recall:    0.6286
Macro F1:        0.6910
Weighted F1:     0.7418

Training time:   0.13 seconds
Inference time:  0.0057 seconds

Classification Report:

                       precision    recall  f1-score   support

               Access       0.92      0.72      0.81      1425
Administrative rights       0.99      0.19      0.32       352
           HR Support       0.78      0.81      0.79      2183
             Hardware       0.61      0.95      0.75      2724
     Internal Project       0.99      0.42      0.59       424
        Miscellaneous       0.84      0.63      0.72      1412
             Purchase       0.98      0.81      0.89       493
              Storage       0.98      0.51      0.67       555

             accuracy                           0.75      9568
            macro avg       0.89      0.63      0.69      9568
         weighted avg       0.80      0.75      0.74      9568



In [19]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy,
        "Macro Precision": precision_macro,
        "Macro Recall": recall_macro,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted,
        "Training Time": training_time,
        "Inference Time": inference_time
    },
    {
        "Model": "Linear SVM",
        "Accuracy": svm_accuracy,
        "Macro Precision": svm_precision_macro,
        "Macro Recall": svm_recall_macro,
        "Macro F1": svm_f1_macro,
        "Weighted F1": svm_f1_weighted,
        "Training Time": svm_training_time,
        "Inference Time": svm_inference_time
    },
    {
        "Model": "Multinomial Naive Bayes",
        "Accuracy": nb_accuracy,
        "Macro Precision": nb_precision_macro,
        "Macro Recall": nb_recall_macro,
        "Macro F1": nb_f1_macro,
        "Weighted F1": nb_f1_weighted,
        "Training Time": nb_training_time,
        "Inference Time": nb_inference_time
    }
])

comparison = comparison.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

display(comparison)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time,Inference Time
0,Linear SVM,0.858696,0.879730,0.845599,0.861356,0.858747,1.358171,0.005790
1,Logistic Regression,0.850753,0.890756,0.814560,0.847017,0.850757,11.222941,0.004729
2,Multinomial Naive Bayes,0.750209,0.885944,0.628559,0.690983,0.741825,0.128424,0.005719


In [20]:
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

comparison.to_csv(
    results_dir / "traditional_ml_baseline_results.csv",
    index=False
)

print(
    "Saved:",
    results_dir / "traditional_ml_baseline_results.csv"
)

Saved: C:\Users\tscou\capstone\results\traditional_ml_baseline_results.csv
